# 01 - Preparação de Dados

Carrega o arquivo `espelhos_acordaos_artigo2026.parquet` e o complementa com:
1. **Inteiro Teor** de cada acórdão, obtido dos ZIPs publicados na Base de Dados Abertos do STJ (dataset `integras-de-decisoes-terminativas-e-acordaos-do-diario-da-justica`).
2. **Dados do espelho** de cada acórdão (tese jurídica, tema, referências legislativas, etc.), obtidos dos JSONs de espelhos publicados no Portal de Dados Abertos do STJ.

- Dados Abertos: https://dadosabertos.web.stj.jus.br/group/jurisprudencia
- Sobre o espelho do acórdão: https://scon.stj.jus.br/docs_internet/jurisprudencia/ajuda/Espelho_do_Acordao_atualizado.pdf
- Sobre a api CKAN: https://docs.ckan.org/en/2.9/api/

Textos:
- No Portal de Dados Abertos, o dataset de espelhos contém a Ementa do Acórdão, enquanto o dataset de integras contém o Inteiro Teor.
- Outros campos textuais do espelho não estão completos ou não estão no padrão que é o objetivo do experimento, por isso não podem der usados como tabela da verdade no experimento
- A ligação entre os dados dos espelhos e das íntegras é feito automaticamente pela chave composta (num_registro, data de publicação, tipo de documento)
- Podem existir espelhos que levam a mais de uma íntegra ou vice-versa, esses dados são descartados

No caso de haver problemas com o download de algum arquivo usando a api, pode-se baixar manualmente o arquivo e colocar na pasta de cache:
- Download manual dos arquivos com as íntegras: https://dadosabertos.web.stj.jus.br/dataset/integras-de-decisoes-terminativas-e-acordaos-do-diario-da-justica
- Download manual dos arquivos com os espelhos: https://dadosabertos.web.stj.jus.br/organization/superior-tribunal-de-justica?tags=jurisprudencia

## Pipeline
```
parquet → set(seq_documento_acordao) + set(num_registro)
         ↓
CKAN → lista de ZIPs (íntegras) + lista de JSONs (espelhos por órgão)
         ↓  (download com cache)
ZIP em memória → filtro por seq_documento → coluna 'texto'
JSON de espelho → filtro por num_registro  → colunas 'teseJuridica', 'tema', etc.
         ↓
parquet enriquecido com texto + campos do espelho
```

## 1. Configuração e Imports

In [1]:
# Importação do pacote utilitário
import os, sys
from pathlib import Path
try:
   sys.path.extend(['../','../src'])
   from util import UtilEnv
   if UtilEnv.carregar_env(pastas=['./','../']):
      print('✅ Variáveis de ambiente carregadas _o/')
except:
    UtilEnv = None
    print('⚠️ Não foi possível importar o pacote para carregar o .env!')


# ─── Arquivos ─────────────────────────────────────────────────────────────────
PASTA_DATA = '../data'
PARQUET_BASE  = os.path.join(PASTA_DATA, 'espelhos_acordaos_artigo2026.parquet')
PARQUET_SAIDA = os.path.join(PASTA_DATA, 'espelhos_acordaos_artigo2026_com_texto.parquet')

# Pasta raiz para cache dos ZIPs e JSONs (evita re-download)
# As subpastas espelhos/, integras/ e metadados_integras/ são criadas automaticamente pela UtilCkan
# Padrão ./downloads_stj
# DOWNLOAD_DIR = Path('downloads_stj')  

# Define se tenta baixar os zips das íntegras e dos espelhos ou apenas consolida o que já existir
CKAN_TIMEOUT = UtilEnv.get_int('CKAN_TIMEOUT',300) if UtilEnv else 600
ATUALIZAR_CACHE_E_MAPAS = UtilEnv.env_true('ATUALIZAR_CACHE_E_MAPAS',True) if UtilEnv else True

if UtilEnv is None:
   print('⚠️ Variáveis de ambiente não carregadas, verfique pacote Util emc ./src!')
print('Configurações:',
      f'CKAN_TIMEOUT: {CKAN_TIMEOUT}',
      f'ATUALIZAR_CACHE_E_MAPAS: {ATUALIZAR_CACHE_E_MAPAS}',
      f'',
      sep='\n')


Env encontrado em ../.env
✅ Variáveis de ambiente carregadas _o/
Configurações:
CKAN_TIMEOUT: 600
ATUALIZAR_CACHE_E_MAPAS: False



## 2. Carregamento do Parquet Base

In [2]:
import pandas as pd
df = pd.read_parquet(PARQUET_BASE)

# Cria id_peca calculado: número de registro . data de publicação . 
# Exemplo: '202202853462.20230601.'
# Esses serão os nomes dos arquivos de extração do experimento
df['id_peca'] = (
    df['registro'].astype(str)
    + '.'
    + df['publicacao'].astype(str).str.replace('-','')
    + '.'
)

print(f'Total de registros : {len(df)}')
print(f'Colunas            : {df.columns.tolist()}')
print(f'Exemplo id_peca    : {df["id_peca"].iloc[0]}')
print()
df.head(3)


Total de registros : 1225
Colunas            : ['ano', 'documento', 'registro', 'publicacao', 'id_peca']
Exemplo id_peca    : 202202853462.20230601.



,ano,documento,registro,publicacao,id_peca
0,2023,188798478,202202853462,2023-06-01,202202853462.20230601.
1,2022,156649445,202201555326,2022-06-20,202201555326.20220620.
2,2022,155998284,202201341093,2022-06-15,202201341093.20220615.


In [3]:
# prepara os filtros para uso de UtilCKan
documentos_necessarios = None
registros_necessarios = None

if 'acordao' in df.columns:
    # ── Lookup: seq_documento_acordao → índices do dataframe (para junção de textos) ──
    seq_para_idx: dict[int, list[int]] = {}
    for idx, val in df['acordao'].dropna().items():
        seq = int(val)
        seq_para_idx.setdefault(seq, []).append(idx)
    documentos_necessarios = set(seq_para_idx.keys())
    print(f'seq_documento_acordao únicos necessários: {len(documentos_necessarios)}')
    print(f'Exemplos: {list(documentos_necessarios)[:5]}')

elif 'registro' in df.columns and 'publicacao' in df.columns:
    # ── Lookup por numero_registro + data_publicacao ──
    if 'tipo_decisao' in df.columns:
        registros_necessarios = set(zip(df['registro'].astype(str), df['publicacao'].astype(str), df['tipo_decisao'].astype(str)))
    else:
        registros_necessarios = set(zip(df['registro'].astype(str), df['publicacao'].astype(str)))
    print(f'registros_necessarios únicos: {len(registros_necessarios)}')
    print(f'Exemplos: {list(registros_necessarios)[:5]}')

elif 'registro' in df.columns:
    registros_necessarios = set(df['registro'].dropna().astype(str).unique())
    print(f'registros_necessarios únicos: {len(registros_necessarios)}')
    print(f'Exemplos: {list(registros_necessarios)[:5]}')


# ── Anos em estudo ──
anos_necessarios = {'2023','2024'}


registros_necessarios únicos: 1225
Exemplos: [('202203430843', '2023-08-24'), ('202202074969', '2023-02-17'), ('202201491023', '2023-12-15'), ('202300658160', '2023-04-28'), ('202203818774', '2023-03-31')]


## 3. Extração via UtilCkan
Instanciação da classe com os filtros exatos de documentos e registros.

In [4]:
import importlib, util_ckan
importlib.reload(util_ckan)
from util_ckan import UtilCkan

ckan = UtilCkan(
    anos              = anos_necessarios if anos_necessarios else None,
    documentos        = documentos_necessarios,
    registros         = registros_necessarios,
    timeout           = CKAN_TIMEOUT,
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,
)

print(f'CKAN      : {ckan.base_url}')
print(f'Cache dir : {ckan.download_dir}')
print()

df_resultado = ckan.gerar_dataset_espelhos(
    incluir_integras = True,
    incluir_ementas  = True,
    incluir_decisoes = True,
)


  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
CKAN      : https://dadosabertos.web.stj.jus.br
Cache dir : downloads_stj

Registros cruzados (espelho × íntegra): 1182


Espelhos:   0%|          | 0/44 [00:00<?, ?it/s]


Registros extraídos: 1183


Extraindo íntegras:   0%|          | 0/123 [00:00<?, ?it/s]

Íntegras extraídas: 1058 / 1058


## 4. Amostra

In [5]:
ckan.exibir_amostra(df_resultado)



Amostra (1182 registros totais, exibindo 2):
═════════════════════════════════════════════════════════════════
  dataDecisao                 : 20230510
  dataPublicacao              : DJE        DATA:01/06/2023
  data_publicacao_iso         : 20230601
  descricaoClasse             : HABEAS CORPUS
  id                          : 000844657
  id_mapa                     : 202202853462.20230601.ACÓRDÃO
  jurisprudenciaCitada        : (RECONHECIMENTO FOTOGRÁFICO - MEIO DE PROVA - INEXISTÊNCIA DE CARÁTER ABSOLUTO)    STJ - <<HC 712781>>-RJ, <<AgRg no HC 730232>>-SP,          <<AgRg no REsp 1969149>>-RS (MEIO DE PROVA - CONTRADIÇÕES - DÚVIDA RAZOÁVEL - IN DUBIO PRO REO)    STJ - <<AgRg no AREsp 1722914>>-DF, <<HC 706735>>-RS,          <<HC 674139>>-SP, <<HC 664537>>-RJ, <<HC 698058>>-SP    STF - [[HC 170117]]
  ministroRelator             : LAURITA VAZ
  nomeOrgaoJulgador           : TERCEIRA SEÇÃO
  numeroRegistro              : 202202853462
  orgao                       : S3
  referenciasL

## 5. Métricas e Salvamento

In [6]:
df_resultado.to_parquet(PARQUET_SAIDA, index=False)
ckan.exibir_metricas(df_resultado, PARQUET_SAIDA)

───────────────────────────────────────────────────────
  ✅  ARQUIVO GERADO
───────────────────────────────────────────────────────
  Arquivo   : ../data/espelhos_acordaos_artigo2026_com_texto.parquet
  Tamanho   : 10.01 MB
  Colunas   : 25  →  ['id', 'numeroRegistro', 'siglaClasse', 'descricaoClasse', 'nomeOrgaoJulgador', 'ministroRelator', 'tipoDeDecisao', 'dataPublicacao', 'dataDecisao', 'teseJuridica', 'tema', 'referenciasLegislativas', 'jurisprudenciaCitada', 'notas', 'termosAuxiliares', 'informacoesComplementares', 'acordaosSimilares', 'ementa', 'decisao', 'id_mapa', 'orgao', 'data_publicacao_iso', 'seq_documento_acordao', 'tem_integra', 'integra']
───────────────────────────────────────────────────────
  📊  COBERTURA DOS DADOS
───────────────────────────────────────────────────────
  Total de registros        :   1182
  Com texto da íntegra      :   1058  (89.5%)
  Sem texto da íntegra      :    124  (10.5%)
  Com dados de espelho      :   1182  (100.0%)
  Sem dados de espelho  